# PRODUCTION

In [2]:
# ==== IMPORTS ====
import pandas as pd
import json
import re

# ==== SETTINGS ====
pd.set_option("display.max_colwidth", None)  # Show full text in cells

# ==== FUNCTIONS ====
# _________________________________________________________
# ---- Functions for loading and rinsing dataframes ----
def load_file(path):
    try:
        df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path, sep=";", encoding="utf-8")
        resolved_path = path
    except FileNotFoundError:
        alt_path = f"../{path}"
        df = pd.read_parquet(alt_path) if alt_path.endswith(".parquet") else pd.read_csv(alt_path, sep=";", encoding="utf-8")
        resolved_path = alt_path
    return df, resolved_path

def load_json(path):
    try:
        with open(path, "r") as f:
            data = json.load(f)
        resolved_path = path
    except FileNotFoundError:
        alt_path = f"../{path}"
        with open(alt_path, "r") as f:
            data = json.load(f)
        resolved_path = alt_path
    return data, resolved_path

def df_info(df, path):
    print(f"DataFrame loaded from: {path}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}\n")

def duplicate_rinse(df,collumn):
    # Duplication check and rinse
    df_rinsed = df.drop_duplicates(subset=[collumn])
    print(f"Rows with duplicated {collumn}: {len(df) - len(df_rinsed)}")
    print(f"Shape after removing duplicated {collumn} rows: {df_rinsed.shape}\n")
    return df_rinsed

def drop_2026(df, year_col="Publication Year", target_year=2026):
    # If df containts column 'Publication Year', drop the rows where 'Publication Year' is 2026
    if target_year in df[year_col].values:
        df = df[df[year_col] != target_year]
    return df

def run_info_n_rinse(df, path, collumn):
    name = path.split("/")[-1].rsplit(".", 1)[0]
    print(f"===== {name} =====")
    df_info(df, path)
    df_rinsed = duplicate_rinse(df, collumn)
    return df_rinsed
# ________________________________________________________

# ________________________________________________________
# ---- Functions for department mapping ----
def _flatten_text(value):
    """Return a flat list of strings from nested str/list/dict structures."""
    out = []
    if value is None:
        return out
    if isinstance(value, str):
        out.extend([v.strip() for v in value.split("|") if v.strip()])
    elif isinstance(value, list):
        for v in value:
            out.extend(_flatten_text(v))
    elif isinstance(value, dict):
        for v in value.values():
            out.extend(_flatten_text(v))
    return out

def _norm(text):
    text = str(text).strip().lower()
    text = re.sub(r"^dtu\s+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def _department_value(dep_item):
    dept = dep_item.get("department")
    if isinstance(dept, dict):
        return dept.get("en") or dept.get("da") or next((str(v) for v in dept.values() if v), None)
    return dept

def build_alias_lookup(dep_items):
    """Build alias -> department lookup from title + sections in dep JSON."""
    alias_lookup = {}
    for item in dep_items:
        department_val = _department_value(item)
        aliases = []
        aliases.extend(_flatten_text(item.get("title")))
        aliases.extend(_flatten_text(item.get("sections")))
        for alias in aliases:
            alias_lookup[_norm(alias)] = department_val
    return alias_lookup

def _match_department_from_text(raw_text, alias_lookup):
    """Try exact match first, then contains match on one text value."""
    if pd.isna(raw_text):
        return pd.NA

    txt_norm = _norm(raw_text)

    if txt_norm in alias_lookup:
        return alias_lookup[txt_norm]

    for alias, dep_val in alias_lookup.items():
        if alias in txt_norm or txt_norm in alias:
            return dep_val

    return pd.NA

def _match_from_affiliations(affiliations, alias_lookup):
    """
    Try matching each affiliations segment from left to right.
    Example format: "segment1|segment2|segment3"
    """
    if pd.isna(affiliations):
        return pd.NA

    parts = [part.strip() for part in str(affiliations).split("|") if part.strip()]
    for part in parts:
        match = _match_department_from_text(part, alias_lookup)
        if pd.notna(match):
            return match

    return pd.NA

def add_department_mapping(
    df,
    dep_items,
    publisher_col="Publisher",
    affiliations_col="Affiliations",
    output_col="Department_new",
):
    """
    Reusable mapping for any dataframe with publisher/affiliations columns.

    Strategy:
    1) Try matching department from publisher.
    2) If unmatched, fallback to affiliations segments split by '|', left -> right.
    """
    required_cols = [publisher_col, affiliations_col]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    alias_lookup = build_alias_lookup(dep_items)
    out_df = df.copy()

    out_df[output_col] = out_df[publisher_col].apply(
        lambda value: _match_department_from_text(value, alias_lookup)
    )

    unmatched_mask = out_df[output_col].isna()
    if unmatched_mask.any():
        out_df.loc[unmatched_mask, output_col] = out_df.loc[unmatched_mask, affiliations_col].apply(
            lambda value: _match_from_affiliations(value, alias_lookup)
        )

    return out_df

def dep_map_checks(df):
    """Department mapping checks"""
    matched_count = df["Department_new"].notna().sum()
    print("Matched rows:", matched_count, "/", len(df))
    #display(df[["Affiliations", "Publisher", "Department_new"]].head(20))

    ### Test for unmatched rows
    if df["Department_new"].isna().any():
        print("Not all rows were matched, the unmatched rows are:")
        print()

    unmatched_publishers = (
        df.loc[df["Department_new"].isna(), "Publisher"]
        .fillna("Missing")
        .astype(str)
        .str.strip()
        .replace("", "Missing")
        .value_counts()
        .rename_axis("Publisher")
        .reset_index(name="Count")
    )

    print(f"Unmatched rows: {unmatched_publishers['Count'].sum()} / {len(df)}")
    display(unmatched_publishers.head(30))

    # Drop unmatched rows
    print("NOTE: Unmatched rows have been dropped from the returned DataFrame.\n")
    return_df = df[df["Department_new"].notna()].reset_index(drop=True)

    return return_df
# ________________________________________________________

# ________________________________________________________
# ---- Export function(s) ----
def export_csv(df, path, name):
    """USAGE:

     export_csv(
    df=df_meta_rinsed,
    path="../[input your path here]/",
    name="[name of file here].csv",
    )
 """

    path_name = path + name
    df.to_csv(path_name, index=False, sep=";", encoding="utf-8")
    print(f"Exported updated DataFrame to: {path_name}")

In [3]:
# ==== MAIN ===

# Load and process metadata and metrics files
#df_meta, path_meta = load_file("Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined.parquet")
df_meta, path_meta = load_file("Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined_filtered.csv")
df_metrics, path_metrics = load_file("Data/extracted_metrics.csv")

# Remove 2026 from meta
df_meta = drop_2026(df_meta, year_col="Publication Year", target_year=2026)

# rinse duplicates based on member_id_ss and member_id_ss_metrics
df_meta_rinsed = run_info_n_rinse(df_meta, path_meta, "primary_member_id_s")
df_metrics_rinsed = run_info_n_rinse(df_metrics, path_metrics, "member_id_ss_metrics")

===== thesis_meta_combined_filtered =====
DataFrame loaded from: ../Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined_filtered.csv
Shape: (6251, 16)
Columns: ['abstract_ts', 'access_ss', 'Timestamp', 'Author', 'ID', 'collection_facet', 'Publication Year', 'format', 'fulltext_availability_facet', 'isolanguage_facet', 'member_id_ss', 'primary_member_id_s', 'Publisher', 'Source', 'source_all_ss', 'Title']

Rows with duplicated primary_member_id_s: 1976
Shape after removing duplicated primary_member_id_s rows: (4275, 16)

===== extracted_metrics =====
DataFrame loaded from: ../Data/extracted_metrics.csv
Shape: (6302, 7)
Columns: ['pdf_file', 'member_id_ss_metrics', 'num_tot_pages', 'num_cont_pages', 'match_trigger', 'num_words_full', 'num_words_cont']

Rows with duplicated member_id_ss_metrics: 0
Shape after removing duplicated member_id_ss_metrics rows: (6302, 7)



In [4]:
# Load helper file: the JSON file with the departments and sections of DTU
dep, dep_path = load_json("../Data/gcp_order/helper_files/department_classification.json")
print(f"Loaded department mapping from: {dep_path}")

# Add new column with department mapping based on publisher and affiliations
df_meta_rinsed = add_department_mapping(
    df=df_meta_rinsed,
    dep_items=dep,
    publisher_col="Publisher",
    affiliations_col="Publisher",   #"Affiliations" if available else reuse the same column as publisher for matching
    output_col="Department_new",
)
# Department mapping checks
df_meta_clean = dep_map_checks(df_meta_rinsed)

df_meta_clean = run_info_n_rinse(df_meta_clean, path_meta, "primary_member_id_s")

Loaded department mapping from: ../Data/gcp_order/helper_files/department_classification.json
Matched rows: 4275 / 4275
Unmatched rows: 0 / 4275


,Publisher,Count


NOTE: Unmatched rows have been dropped from the returned DataFrame.

===== thesis_meta_combined_filtered =====
DataFrame loaded from: ../Data/gcp_order/dtu_findit/master_thesis_meta/thesis_meta_combined_filtered.csv
Shape: (4275, 17)
Columns: ['abstract_ts', 'access_ss', 'Timestamp', 'Author', 'ID', 'collection_facet', 'Publication Year', 'format', 'fulltext_availability_facet', 'isolanguage_facet', 'member_id_ss', 'primary_member_id_s', 'Publisher', 'Source', 'source_all_ss', 'Title', 'Department_new']

Rows with duplicated primary_member_id_s: 0
Shape after removing duplicated primary_member_id_s rows: (4275, 17)



In [5]:
relevant_meta_columns = ["abstract_ts", "Author", "Publication Year", "primary_member_id_s", "Title", "Department_new"]

# merging the rinsed metadata and metrics dataframes on member_id_ss (df_meta_rinsed) and 'member_id_ss_metrics' (master_thesis_metrics_analysis) for collumns in relevant_meta_columns in df_meta_rinsed
master_thesis_metrics_analysis = pd.merge(
    df_metrics_rinsed,
    df_meta_clean[relevant_meta_columns],
    left_on="member_id_ss_metrics",
    right_on="primary_member_id_s",
    how="inner",
)

# add author count to the master_thesis_metrics_analysis DataFrame as a new column 'author_count'
master_thesis_metrics_analysis['author_count'] = master_thesis_metrics_analysis['Author'].str.split('|').str.len()

print(f"Collumns of DataFrame:\n {list(master_thesis_metrics_analysis.columns)}\n")

#display(master_thesis_metrics_analysis.head(3))
print(f"Final merged DataFrame shape: {master_thesis_metrics_analysis.shape}\n")

export_DF = input("Do you want to export the merged DataFrame as a CSV file? (y/n): ").strip().lower()
if export_DF == "y":
    export_csv(
        df=master_thesis_metrics_analysis,
        path="../Data/gcp_order/dtu_findit/extraction_and_processing/",
        name="master_thesis_metrics_analysis.csv",
    )
else:
    print("Export skipped.")

Collumns of DataFrame:
 ['pdf_file', 'member_id_ss_metrics', 'num_tot_pages', 'num_cont_pages', 'match_trigger', 'num_words_full', 'num_words_cont', 'abstract_ts', 'Author', 'Publication Year', 'primary_member_id_s', 'Title', 'Department_new', 'author_count']

Final merged DataFrame shape: (4254, 14)



Export skipped.


# DEVELOPMENT

## TO DO: 

- [x] get pdf page count (incl/excl. appendix) 
- [x] get word count 
- [x] rewrite for GCP and append results to csv file
- [x] populate df ```master_thesis_metrics_analysis``` with data from metrics and meta that is relevant for analysis
- [] explore the dataset ````Data/gcp_order/dtu_findit/extraction_and_processing/master_thesis_metrics_analysis.csv````
- [] get supervisor(s)

# ARCHIVES